# Guildmaster-AI Playground

Interactive notebook to explore the Guildmaster-AI framework.

Make sure your `.env` file has `OPENROUTER_API_KEY` set.

In [1]:
import os

from dotenv import load_dotenv

# Load .env from project root (works whether kernel runs from repo root or examples/)
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env")) or load_dotenv(".env")

# Ensure we're working from the project root for file paths
if os.path.basename(os.getcwd()) == "examples":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()
print(f"Project root: {PROJECT_ROOT}")
print(f"API key loaded: {'yes' if os.getenv('OPENROUTER_API_KEY') else 'no'}")

Project root: /Users/kobros/repos/private/guildmaster-ai
API key loaded: yes


## 1. Basic Quest — Simple Q&A

The simplest usage: create a Guild with an OpenRouter LLM and a GeneralAdventurer,
then post a quest.

In [2]:
from guildmaster_ai import GuildBuilder
from guildmaster_ai.adventurers.general_adventurer import GeneralAdventurer

guild = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(GeneralAdventurer)
    .build()
)

print("Guild created!")
print(f"Roster: {[p.name or p.id for p in guild.roster]}")

[guildmaster.guildmaster] INFO: Registered adventurer '5cda8608-b2cd-4755-ba14-36dcb9f9bc3d' with talents ['general']


Guild created!
Roster: ['5cda8608-b2cd-4755-ba14-36dcb9f9bc3d']


In [3]:
result = await guild.post_quest(
    "What are the 3 key differences between Python and Rust? Be concise."
)

print(f"Success: {result.success}")
print("---")
print(result.summary)

[guildmaster.guild] INFO: === New quest request: What are the 3 key differences between Python and Rust? Be concise. ===
[guildmaster.guild] INFO: Phase 1: Planning
[guildmaster.receptionist] INFO: Intake started for request: What are the 3 key differences between Python and Rust? Be concise.
[guildmaster.receptionist] INFO: Refining quest draft via LLM
[guildmaster.receptionist] INFO: Intake complete: 'Research: Compare Python and Rust Programming Languages'
[guildmaster.guildmaster] INFO: Checking feasibility for 'Research: Compare Python and Rust Programming Languages' (requires: ['Programming Knowledge', 'Research', 'Technical Writing'])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=1 missing=[]
[guildmaster.guild] INFO: Quest created: bbfb0bd6 ('Research: Compare Python and Rust Programming Languages')
[guildmaster.guild] INFO: Phase 2: Execution
[guildmaster.guildmaster] INFO: Assigning quest bbfb0bd6: 'Research: Compare Python and Rust Programming Languages'

Success: True
---
I'll conduct research to identify the three most significant differences between Python and Rust programming languages, focusing on the areas you specified.

## Research Findings: Three Most Significant Differences Between Python and Rust

### 1. **Memory Management and Safety Paradigms**

**Python:**
- Uses automatic garbage collection for memory management
- Memory safety is handled at runtime through the interpreter
- Developers don't need to explicitly manage memory allocation/deallocation
- Potential for memory leaks and runtime errors related to memory access

**Rust:**
- Implements a unique ownership system with compile-time memory safety guarantees
- No garbage collector - memory is automatically freed when variables go out of scope
- Prevents common memory errors (null pointer dereferences, buffer overflows, use-after-free) at compile time
- Zero-cost abstractions - safety features don't impact runtime performance

**Significance:** This represents a fundamen

## 2. Using the LLM Directly

You can use the LLM layer directly without the full quest lifecycle.
`guild_complete()` is a simple helper for one-shot completions.

In [4]:
from guildmaster_ai.llm import create_chat_model, guild_complete

llm = create_chat_model("openrouter")

answer = await guild_complete(
    llm,
    system="You are a helpful assistant that responds in haiku format.",
    user="Tell me about async programming.",
)
print(answer)

Concurrent tasks run,
Without blocking the main thread—
Efficiency flows.

Callbacks, promises,
Async-await patterns help
Code stay responsive.

I/O waits no more,
Other work proceeds meanwhile—
Time becomes precious.


## 3. Equipping Weapons (Tools)

Adventurers can use weapons (tools) during quests. The LLM decides when to call them.
Here we equip the `FileReadWeapon` so the adventurer can read files.

In [5]:
from guildmaster_ai.weapons.file_read import FileReadWeapon

adventurer = GeneralAdventurer(name="FileReader")
adventurer.equip_weapon(FileReadWeapon())

guild_with_tools = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(adventurer)
    .build()
)

# Use absolute path so it works regardless of kernel working directory
toml_path = os.path.join(PROJECT_ROOT, "pyproject.toml")

result = await guild_with_tools.post_quest(
    f"Read the file '{toml_path}' and tell me what Python version is required "
    "and list all the core dependencies. Be concise."
)

print(f"Success: {result.success}")
print("---")
print(result.summary)

[guildmaster.guildmaster] INFO: Registered adventurer 'FileReader' with talents ['general', 'file_read']
[guildmaster.guild] INFO: === New quest request: Read the file '/Users/kobros/repos/private/guildmaster-ai/pyproject.toml' and tell me what Python ve ===
[guildmaster.guild] INFO: Phase 1: Planning
[guildmaster.receptionist] INFO: Intake started for request: Read the file '/Users/kobros/repos/private/guildmaster-ai/pyproject.toml' and tell me what Python ve
[guildmaster.receptionist] INFO: Refining quest draft via LLM
[guildmaster.receptionist] INFO: Intake complete: 'Analyze Python Project Configuration File'
[guildmaster.guildmaster] INFO: Checking feasibility for 'Analyze Python Project Configuration File' (requires: ['file_reading', 'python_configuration_analysis', 'toml_parsing'])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=1 missing=[]
[guildmaster.guild] INFO: Quest created: 3dc411d2 ('Analyze Python Project Configuration File')
[guildmaster.guild] INFO

Success: True
---
## Analysis of pyproject.toml

### Python Version Requirements
- **Minimum Python Version**: `>=3.11`
- The project targets Python 3.11 as confirmed by both the `requires-python` field and the tool configurations (ruff and mypy)

### Core Dependencies
The project has the following core dependencies that are installed by default:

1. **Pydantic Stack**:
   - `pydantic>=2.0,<3.0` - Data validation and settings management
   - `pydantic-settings>=2.0,<3.0` - Settings management using Pydantic

2. **Database & Storage**:
   - `aiosqlite>=0.20.0` - Asynchronous SQLite database interface
   - `chromadb>=0.5.0` - Vector database for embeddings

3. **LangChain Framework**:
   - `langchain>=1.0` - Core LangChain framework
   - `langchain-core>=1.0` - Core LangChain components
   - `langchain-openai>=1.0` - OpenAI integration for LangChain

4. **Utilities**:
   - `python-dotenv>=1.2.2` - Environment variable management from .env files

### Optional Dependencies
The project incl

## 4. Armor (Guardrails)

Armor provides pre/post processing guardrails. The `ContentFilterArmor`
blocks messages matching forbidden patterns.

In [6]:
from guildmaster_ai.armor.content_filter import ContentFilterArmor

guarded_adventurer = GeneralAdventurer(name="GuardedAdventurer")
guarded_adventurer.wear_armor(
    ContentFilterArmor(blocked_patterns=["password", "secret"])
)

guild_guarded = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(guarded_adventurer)
    .build()
)

# This quest should work fine
result = await guild_guarded.post_quest("What is 2 + 2? Answer in one word.")
print(f"Clean quest - Success: {result.success}")
print(f"Answer: {result.summary}")

print("\n---\n")

# This quest should be blocked by the content filter
try:
    result = await guild_guarded.post_quest("What is my password?")
except RuntimeError as e:
    print(f"Blocked! {e}")

[guildmaster.guildmaster] INFO: Registered adventurer 'GuardedAdventurer' with talents ['general', 'content_filter']
[guildmaster.guild] INFO: === New quest request: What is 2 + 2? Answer in one word. ===
[guildmaster.guild] INFO: Phase 1: Planning
[guildmaster.receptionist] INFO: Intake started for request: What is 2 + 2? Answer in one word.
[guildmaster.receptionist] INFO: Refining quest draft via LLM
[guildmaster.receptionist] INFO: Intake complete: 'What is 2 + 2? Answer in one word.'
[guildmaster.guildmaster] INFO: Checking feasibility for 'What is 2 + 2? Answer in one word.' (requires: [])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=1 missing=[]
[guildmaster.guild] INFO: Quest created: 7cfe7386 ('What is 2 + 2? Answer in one word.')
[guildmaster.guild] INFO: Phase 2: Execution
[guildmaster.guildmaster] INFO: Assigning quest 7cfe7386: 'What is 2 + 2? Answer in one word.'
[guildmaster.guild] INFO: Quest leader: GuardedAdventurer
[guildmaster.adventurer.genera

Clean quest - Success: True
Answer: Four

---



[guildmaster.receptionist] INFO: Intake complete: 'What is my password?'
[guildmaster.guildmaster] INFO: Checking feasibility for 'What is my password?' (requires: [])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=1 missing=[]
[guildmaster.guild] INFO: Quest created: 209c3b2c ('What is my password?')
[guildmaster.guild] INFO: Phase 2: Execution
[guildmaster.guildmaster] INFO: Assigning quest 209c3b2c: 'What is my password?'
[guildmaster.guild] INFO: Quest leader: GuardedAdventurer
[guildmaster.adventurer.general] INFO: Starting quest 209c3b2c: 'What is my password?'


Blocked! Armor content_filter blocked input: Content blocked: matched pattern 'password'


## 5. Guard (LLM-as-Judge)

Enable the Guard to have an LLM evaluate quest results for quality and safety.

In [7]:
guild_with_guard = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(GeneralAdventurer)
    .with_guard()
    .build()
)

result = await guild_with_guard.post_quest(
    "Write a short Python function that checks if a number is prime."
)

print(f"Success: {result.success}")
print("---")
print(result.summary)

[guildmaster.guildmaster] INFO: Registered adventurer '9cc65f44-fbee-4cff-ad71-3a28dcb149fc' with talents ['general']
[guildmaster.guild] INFO: Guard enabled: guard
[guildmaster.guild] INFO: === New quest request: Write a short Python function that checks if a number is prime. ===
[guildmaster.guild] INFO: Phase 1: Planning
[guildmaster.receptionist] INFO: Intake started for request: Write a short Python function that checks if a number is prime.
[guildmaster.receptionist] INFO: Refining quest draft via LLM
[guildmaster.receptionist] INFO: Intake complete: 'Create Prime Number Checker Function'
[guildmaster.guildmaster] INFO: Checking feasibility for 'Create Prime Number Checker Function' (requires: ['Python Programming', 'Algorithm Design', 'Mathematical Logic'])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=1 missing=[]
[guildmaster.guild] INFO: Quest created: 3b9096ce ('Create Prime Number Checker Function')
[guildmaster.guild] INFO: Phase 2: Execution
[guildmas

Success: True
---
Here's an efficient Python function to determine if a number is prime:

```python
def is_prime(n):
    """
    Determines whether a given number is prime.
    
    A prime number is a natural number greater than 1 that has no positive 
    divisors other than 1 and itself.
    
    Args:
        n (int): The number to check for primality
        
    Returns:
        bool: True if the number is prime, False otherwise
    """
    # Handle edge cases
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    
    # Check odd divisors up to sqrt(n)
    i = 3
    while i * i <= n:
        if n % i == 0:
            return False
        i += 2
    
    return True


# Alternative implementation with more explicit optimizations
def is_prime_optimized(n):
    """
    Optimized version with additional early checks for better performance.
    """
    # Handle edge cases
    if n < 2:
        return False
    if n in (2, 3)

## 6. Custom Adventurer

Create your own adventurer by subclassing `BaseAdventurer`.

In [8]:
from guildmaster_ai.adventurers.base_adventurer import BaseAdventurer
from guildmaster_ai.core.messages import QuestResult
from guildmaster_ai.core.quest import Quest


class PirateAdventurer(BaseAdventurer):
    """An adventurer that speaks like a pirate!"""

    @property
    def system_prompt(self) -> str:
        return (
            "You are a pirate adventurer! You speak in pirate dialect "
            "(arr, matey, ye, etc.) but still provide accurate and helpful answers. "
            "Keep responses concise \u2014 2-3 sentences max."
        )

    async def execute(self, quest: Quest) -> QuestResult:
        self._reset_conversation()
        self._add_user_message(quest.description)
        response = await self._call_llm()
        return QuestResult(
            sender=self.name or self.id,
            quest_id=quest.id,
            success=True,
            summary=response.content,
        )


guild_pirate = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(PirateAdventurer(name="Captain Hook"))
    .build()
)

result = await guild_pirate.post_quest("Explain what a REST API is.")
print(result.summary)

[guildmaster.guildmaster] INFO: Registered adventurer 'Captain Hook' with talents ['general']
[guildmaster.guild] INFO: === New quest request: Explain what a REST API is. ===
[guildmaster.guild] INFO: Phase 1: Planning
[guildmaster.receptionist] INFO: Intake started for request: Explain what a REST API is.
[guildmaster.receptionist] INFO: Refining quest draft via LLM
[guildmaster.receptionist] INFO: Intake complete: 'Explain what a REST API is.'
[guildmaster.guildmaster] INFO: Checking feasibility for 'Explain what a REST API is.' (requires: [])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=1 missing=[]
[guildmaster.guild] INFO: Quest created: 0cb1698d ('Explain what a REST API is.')
[guildmaster.guild] INFO: Phase 2: Execution
[guildmaster.guildmaster] INFO: Assigning quest 0cb1698d: 'Explain what a REST API is.'
[guildmaster.guild] INFO: Quest leader: Captain Hook
[guildmaster.guild] INFO: Phase 3: Verification
[guildmaster.guild] INFO: Quest 0cb1698d completed s

Arr, matey! A REST API be like a treasure map that lets different ships (applications) communicate with each other across the digital seas. It uses simple HTTP requests like GET, POST, PUT, and DELETE to fetch, send, or modify data, just like how ye might signal other vessels with flags! Think of it as a standardized way for programs to parley and share their digital booty, savvy?


## 7. Pass Any LangChain Model

You can pass any LangChain `BaseChatModel` directly to the builder.
This works with any LangChain-compatible model.

In [9]:
from guildmaster_ai.llm.openrouter import ChatOpenRouter

# Create a custom-configured LLM (different model, lower temperature)
custom_llm = ChatOpenRouter(
    model="meta-llama/llama-4-scout",
    temperature=0.3,
    max_tokens=256,
)

guild_custom = (
    GuildBuilder()
    .with_llm_provider(custom_llm)  # pass the LangChain model directly
    .register_adventurer(GeneralAdventurer)
    .build()
)

result = await guild_custom.post_quest("What is the capital of France? Answer in one word.")
print(result.summary)

[guildmaster.guildmaster] INFO: Registered adventurer '222004bc-1b21-4e6d-9247-f0b43d015d95' with talents ['general']
[guildmaster.guild] INFO: === New quest request: What is the capital of France? Answer in one word. ===
[guildmaster.guild] INFO: Phase 1: Planning
[guildmaster.receptionist] INFO: Intake started for request: What is the capital of France? Answer in one word.
[guildmaster.receptionist] INFO: Refining quest draft via LLM
[guildmaster.receptionist] INFO: Intake complete: 'French Capital'
[guildmaster.guildmaster] INFO: Checking feasibility for 'French Capital' (requires: [])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=1 missing=[]
[guildmaster.guild] INFO: Quest created: 44662e92 ('French Capital')
[guildmaster.guild] INFO: Phase 2: Execution
[guildmaster.guildmaster] INFO: Assigning quest 44662e92: 'French Capital'
[guildmaster.guild] INFO: Quest leader: 222004bc-1b21-4e6d-9247-f0b43d015d95
[guildmaster.adventurer.general] INFO: Starting quest 4466

Paris.


## 8. Guild Info & Quest Tracking

Inspect the guild state, look up quests by UUID, and see the full lifecycle history.

In [10]:
# Build a guild with a well-equipped adventurer
equipped = GeneralAdventurer(name="Scout")
equipped.equip_weapon(FileReadWeapon())

guild_inspect = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(equipped)
    .build()
)

# Before any quests
print("=== Guild Info ===")
print(repr(guild_inspect))
print(f"Guild ID: {guild_inspect.info.id}")
print()

for p in guild_inspect.info.adventurers:
    print(f"Adventurer: {p.name}")
    print(f"  Talents: {p.talents}")
    print(f"  Weapons: {p.weapons}")
    print(f"  Armor:   {p.armor}")

# Run two quests
r1 = await guild_inspect.post_quest("What is 1 + 1? Answer with just the number.")
r2 = await guild_inspect.post_quest("Capital of Japan? One word.")

print("\n=== After Quests ===")
info = guild_inspect.info
print(f"Total: {info.total_quests}  Completed: {info.completed}  Failed: {info.failed}")

# List all quests
print("\n=== All Quests ===")
for q in guild_inspect.quests:
    print(f"  [{q.id[:8]}...] {q.title} — {q.status.value}")

# Look up a quest by UUID and inspect its history
quest_id = guild_inspect.quests[0].id
q = guild_inspect.get_quest(quest_id)
r = guild_inspect.get_result(quest_id)

print(f"\n=== Quest Detail: {q.title} ===")
print(f"ID:     {q.id}")
print(f"Status: {q.status.value}")
print(f"Rank:   {q.rank.name}")
print(f"Result: {r.summary if r else 'N/A'}")
print("History:")
for h in q.history:
    fr = h.payload.get("from", "")
    to = h.payload.get("to", "")
    print(f"  {fr} -> {to}  (by {h.actor})")

[guildmaster.guildmaster] INFO: Registered adventurer 'Scout' with talents ['general', 'file_read']
[guildmaster.guild] INFO: === New quest request: What is 1 + 1? Answer with just the number. ===
[guildmaster.guild] INFO: Phase 1: Planning
[guildmaster.receptionist] INFO: Intake started for request: What is 1 + 1? Answer with just the number.
[guildmaster.receptionist] INFO: Refining quest draft via LLM


=== Guild Info ===
Guild(id='4cbc02cd-c4ff-41a3-b1a7-ea9a7eb60227', adventurers=1, completed=0, failed=0, in_progress=0)
Guild ID: 4cbc02cd-c4ff-41a3-b1a7-ea9a7eb60227

Adventurer: Scout
  Talents: ['general', 'file_read']
  Weapons: ['file_read']
  Armor:   []


[guildmaster.receptionist] INFO: Intake complete: 'What is 1 + 1? Answer with just the number.'
[guildmaster.guildmaster] INFO: Checking feasibility for 'What is 1 + 1? Answer with just the number.' (requires: [])
[guildmaster.guildmaster] INFO: Feasibility: feasible=True matched=1 missing=[]
[guildmaster.guild] INFO: Quest created: d23b63ae ('What is 1 + 1? Answer with just the number.')
[guildmaster.guild] INFO: Phase 2: Execution
[guildmaster.guildmaster] INFO: Assigning quest d23b63ae: 'What is 1 + 1? Answer with just the number.'
[guildmaster.guild] INFO: Quest leader: Scout
[guildmaster.adventurer.general] INFO: Starting quest d23b63ae: 'What is 1 + 1? Answer with just the number.'
[guildmaster.adventurer.general] INFO: Quest d23b63ae completed in 1 iteration(s)
[guildmaster.guild] INFO: Phase 3: Verification
[guildmaster.guild] INFO: Quest d23b63ae completed successfully
[guildmaster.guild] INFO: Phase 4: Archival
[guildmaster.librarian] INFO: Archiving quest d23b63ae: 'What is 


=== After Quests ===
Total: 2  Completed: 0  Failed: 0

=== All Quests ===
  [5e3a0538...] Trivia Challenge: Identify Japan's Capital — archived
  [d23b63ae...] What is 1 + 1? Answer with just the number. — archived

=== Quest Detail: Trivia Challenge: Identify Japan's Capital ===
ID:     5e3a0538-08cf-4b1d-a0e1-ac682f870b05
Status: archived
Rank:   E
Result: Tokyo
History:
  draft -> posted  (by quest_board)
  posted -> assigned  (by guildmaster)
  assigned -> in_progress  (by Scout)
  in_progress -> completed  (by guildmaster)
  completed -> archived  (by librarian)
